# ⭐️ DL Week Workshop: Building a Smart PDF ChatBot

In this notebook, we will learn how to create a Smart PDF ChatBot. Follow the instructions in this notebook and run each cell. Try out the exercises marked by "TO DO" and "FIX ME" as you follow along in this workshop.

For more resources, you can find curated blogs and reading materials at the end of this notebook.



## 🛠️ Set up and environment configuration

### Load required libraries

In [1]:
!pip install -q langchain-core langchain-community langchain-text-splitters pypdf sentence-transformers faiss-cpu pdfplumber


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### Load API Keys

It's best practice to store sensitive information like API tokens securely. Google Colab provides `userdata` for this purpose. The code below will then securely retrieve it.

#### How to get your API Key:
**1. Sign In / Sign Up**

Go to [huggingface.co](https://huggingface.co/) and log in. If you don't have an account, create a free one.

**2. Navigate to Access Tokens**

Click your profile icon (top-right corner) → **Settings** → **Access Tokens** (left sidebar). Or go directly: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

**3. Create a New Token**

Click **+ Create new token**.

**4. Configure Your Token**

- **Name** — Give it something descriptive so you know what it's for (e.g., `dlweek_workshop`).
- **Token Type** — For this workshop, select **Fine-grained**, then enable the **Make calls to Inference Providers** permission. This gives you only the access you need, nothing more.

**5. Generate and Copy**

Click **Create token**. Copy the token value immediately — Hugging Face will not show it to you again.

**6. Add token to Colab secrets manager.**

Look for the "🔑" icon in the left panel and name it `HF_TOKEN`.

In [ ]:
from google.colab import userdata

# Get the Hugging Face token from Colab's secrets manager
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("Hugging Face token successfully retrieved.")
except Exception as e:
    print(f"Could not retrieve HF_TOKEN from userdata. Please ensure it's set in Colab secrets. Error: {e}")
    HF_TOKEN = input("Please enter your Hugging Face token: ") # Fallback to input if not found in userdata

### Load required libraries

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from huggingface_hub import InferenceClient
from typing import List, Dict, Optional, Tuple, Any
from dataclasses import dataclass, asdict
import re
import pdfplumber

## 1️⃣ Phase 1: RAG Pipeline

### 1. Download PDF
First, we will download the sample PDF we will be using.

#### PM Wong's Budget Statement 2026
The document that we'll be working with is the Singapore Budget Statement 2026
Recently, Prime Minister and Minister for Finance, Mr Lawrence Wong, delivered Singapore's FY2026 Budget Statement on 12 February 2026 in Parliament. This is a 82 page government document that outlines what our Prime Minister announced in Parliament that day. During his speech, PM Wong talked about plans to address an array of issues to which for this workshop we will be uncovering what they are with the help of RAG.

We chose this document for three reasons:

1. Immediately familiar to a Singaporean audience. The Budget covers topics every participant already knows — CPF, SkillsFuture, housing, cost of living, and fiscal policy. There's no need to spend time unpacking domain jargon. We hope that with this, everyone can focus entirely on evaluating the pipeline's behaviour, not decoding the content.

2. Contains specific, verifiable facts. PM Wong's speech is packed with concrete, attributable claims that most of us Singaporeans are familiar with — CPF top-ups for seniors, a FY2026 growth target at the higher end of 2–3%, AI investment missions, and more. This makes it ideal for Phase 3: participants can judge for themselves whether the chatbot's cited answers are accurate and well-supported, without needing to look anything up.

3. 82 pages of dense, structured content. Long enough to stress-test retrieval — there's real signal buried across many sections — but structured enough (chapters, numbered paragraphs) that chunking and citation behaviour is observable and discussable.



In [ ]:
import gdown

google_drive_url = 'https://drive.google.com/uc?export=download&id=16-AQAFL7gzXJW39O3DOsFUiGFzX6cp4Z'
output_filename = 'SG_Budget_Statement_2026.pdf'

gdown.download(google_drive_url, output_filename)

print(f'Downloaded: {output_filename}')
print('Document: Singapore Budget Statement 2026')
print('Source: Singapore Ministry of Finance')

### 2. Load PDF
We will load the PDF and inspect the document contents.

#### Why Use LangChain's PDF Loader?
- Handles PDF parsing complexity
- Returns standardised Document objects
- Integrates seamlessly with the rest of our RAG pipeline
- Easy to swap for other loaders (CSV, websites, JSON, etc.)

In [ ]:
PDF_PATH = "/content/SG_Budget_Statement_2026.pdf"

def load_pdf(file_path):
    """Load a PDF and extract text."""

    # use LangChain's PDF Loader to load the documents
    loader = PyPDFLoader(file_path)
    # get Document objects
    documents = loader.load()
    print(f"✅ Loaded {len(documents)} pages from {file_path}!")
    return documents

# Load the PDF
documents = load_pdf(PDF_PATH)

#Inspect
print("\n--- Sample of First Page ---")
print(f"Page 0 metadata: {documents[0].metadata}")
print(f"Page 0 content preview (first 300 chars):")
print(documents[0].page_content[:300])

### 3. Chunk PDF

#### Why chunk text?
PDFs can be very long. We split them into smaller, overlapping chunks because:

1. **Token Limits**: LLMs have a maximum token limit (e.g., 4k–128k tokens)
2. **Relevance**: Smaller chunks are more semantically focused
3. **Retrieval Quality**: Easier to find exactly relevant information
4. **Embedding Efficiency**: Faster to embed and search

#### Chunking Strategy: RecursiveCharacterTextSplitter
This splits text recursively by priority:
- `"\n\n"` — paragraph breaks *(preferred)*
- `"\n"` — line breaks
- `" "` — word boundaries
- `""` — individual characters *(last resort)*

This preserves semantic structure as much as possible while respecting the size limit.

---

#### ✏️ Exercise 1: Choose Your Chunking Parameters

| Parameter | What it controls | Rule of thumb |
|---|---|---|
| `CHUNK_SIZE` | Maximum characters per chunk | 200–1500 depending on document type |
| `CHUNK_OVERLAP` | Characters shared between consecutive chunks | 15–20% of `CHUNK_SIZE` |

**Three presets to try:**

| Preset | `CHUNK_SIZE` | `CHUNK_OVERLAP` |
|---|---|---|
| Small | 200 | 40 |
| Medium | 500 | 100 |
| Large | 1000 | 200 |

Pick a preset (or your own values), fill them in below, and run the cell. Observe: **how many chunks did you get?**

In [ ]:
# ✏️ Fill in your chosen values
# Preset options:
#   Small  → CHUNK_SIZE=200,  CHUNK_OVERLAP=40
#   Medium → CHUNK_SIZE=500,  CHUNK_OVERLAP=100
#   Large  → CHUNK_SIZE=1000, CHUNK_OVERLAP=200

CHUNK_SIZE = None  # TODO: replace with your chosen value
CHUNK_OVERLAP = None  # TODO: replace with your chosen value (aim for 15-20% of CHUNK_SIZE)

# Input validation
assert CHUNK_SIZE is not None, "❌ Please set CHUNK_SIZE to a number (e.g. 500)"
assert CHUNK_OVERLAP is not None, "❌ Please set CHUNK_OVERLAP to a number (e.g. 100)"
assert CHUNK_OVERLAP < CHUNK_SIZE, "❌ CHUNK_OVERLAP must be less than CHUNK_SIZE"

def chunk_documents(documents, chunk_size=500, chunk_overlap=100):
    """Split documents into smaller chunks."""
    print(f"Splitting into chunks (size={chunk_size}, overlap={chunk_overlap})")

    # Use text splitter to chunk documents
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    chunks = text_splitter.split_documents(documents)
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks from our PDF documents
chunks = chunk_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

In [ ]:
# 👀 Observe: what does a chunk actually look like?
avg_len = sum(len(c.page_content) for c in chunks) // len(chunks)
print(f"Total chunks: {len(chunks)}")
print(f"Average chunk length: {avg_len} chars")

print("\n" + "=" * 60)
print("CHUNK 0:")
print("=" * 60)
print(chunks[0].page_content)

print("\n" + "=" * 60)
print("CHUNK 1 (look for overlap with the end of Chunk 0):")
print("=" * 60)
print(chunks[1].page_content)

print("\n💡 The repeated text between Chunk 0 and Chunk 1 is the overlap — intentional continuity.")

### 4. Vector Store

#### Embeddings are **numerical representations** of text that capture **semantic meaning** as vectors.

#### Example
```
Text: "The perceptron is a machine learning algorithm"
Embedding: [0.12, -0.45, 0.78, 0.23, -0.56, ...]  (384 dimensions)

Text: "Machine learning algorithms like perceptrons"
Embedding: [0.11, -0.44, 0.79, 0.24, -0.57, ...]  (SIMILAR!)
```

#### Why Embeddings?
1. **Semantic Similarity**: Similar text → similar vectors
2. **Fast Search**: Can compute similarity in milliseconds
3. **Geometric Properties**: Can use math operations on the vectors

#### Our Embedding Model: all-MiniLM-L6-v2
- **Size**: 384 dimensions
- **Speed**: Very fast (~13M parameters)
- **Open Source**: Free to use locally or via HuggingFace

#### Our Vector Database: Facebook AI Similarity Search (**FAISS**)
- Stores vectors efficiently
- Searches millions of vectors in milliseconds
- Can be saved/loaded from disk


In [ ]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
FAISS_INDEX_PATH = "faiss_index"

def create_vector_store(chunks, embedding_model_name="all-MiniLM-L6-v2"):
    """Embed chunks and store in FAISS vector database."""
    print(f"Creating embeddings using: {embedding_model_name}")

    # Load embedding model
    embeddings = HuggingFaceEmbeddings(
        model_name=embedding_model_name,
        model_kwargs={"token": HF_TOKEN}
    )

    print("Building FAISS vector store...")
    # Create vector store
    vector_store = FAISS.from_documents(chunks, embeddings)
    print("✅ Vector store created successfully")
    return vector_store

def save_vector_store(vector_store, save_path="faiss_index"):
    """Save FAISS index to disk."""
    vector_store.save_local(save_path)
    print(f"✅ Vector store saved to: {save_path}")

def load_vector_store(load_path="faiss_index", embedding_model_name="all-MiniLM-L6-v2"):
    """Load a previously saved FAISS index."""
    print(f"Loading vector store from: {load_path}")
    embeddings = HuggingFaceEmbeddings(
        model_name=embedding_model_name,
        model_kwargs={"token": HF_TOKEN}
    )
    vector_store = FAISS.load_local(
        load_path, embeddings, allow_dangerous_deserialization=True
    )
    print("✅ Vector store loaded")
    return vector_store

In [ ]:
# Create and save our vector store
print("CREATING VECTOR STORE FROM CHUNKS")

vector_store = create_vector_store(chunks, EMBEDDING_MODEL)
save_vector_store(vector_store, FAISS_INDEX_PATH)

### 5. Retrieve from Vector Store

#### Retrieval is the process of **finding the most relevant chunks** from your vector store based on a query.

#### How It Works

1. **Query Embedding**: Convert user's question to an embedding using the same model
2. **Similarity Search**: Compare query embedding to all chunk embeddings
3. **Return Top-K**: Return the K most similar chunks
4. **How It Ranks**: Uses **cosine similarity** — measures angle between vectors
   - Similarity = 1.0 → identical meaning
   - Similarity = 0.0 → unrelated
   - Similarity = -1.0 → opposite meaning

#### Important Caveats

- **Similarity ≠ Relevance**: A high similarity score doesn't guarantee the chunk answers your question
- **Query Quality Matters**: Vague queries give vague results
  - ❌ Bad: `"Tell me about the budget"`
  - ✅ Good: `"What did PM Wong announce about CPF top-ups for seniors?"`
- **Top-K Trade-off**:

| k value | Effect |
|---|---|
| 1–2 | Very focused, may miss context |
| 3–5 | Good balance — recommended |
| 7–10 | Comprehensive but introduces noise |
| >10 | Too much context, LLM may get confused |

---

#### ✏️ Exercise 2: Choose k and Inspect Your Retrieval

Two things to fill in below:
1. **`TOP_K_RESULTS`** — how many chunks to retrieve
2. **`test_query`** — your own question about the Singapore Budget 2026

After running, read the retrieved chunks carefully. Ask yourself: **does this context actually contain what's needed to answer your question?**

In [ ]:
TOP_K_RESULTS = 5  # TODO: replace with your chosen value (try 1, 3, 5, or 10)

# Input validation
assert TOP_K_RESULTS is not None, "Please set TOP_K_RESULTS to a number (e.g. 5)"

def retrieval(vector_store, query, k=3):
    """Search the vector store for the most similar chunks to a query."""
    print(f"Retrieving top {k} chunks...\n")
    # Use similarity search and retrieve top k chunks
    results = vector_store.similarity_search(query, k=k)
    return results


In [ ]:
# ✏️ Type your own question about the Singapore Budget 2026
test_query = # TODO: replace with your own question (e.g. "What did PM Wong say about CPF top-ups?")

# Input validation
assert test_query is not None, "Please set test_query to a question string"

# Run retrieval
print(f"🔍 Query: {test_query}")
retrieved_chunks = retrieval(vector_store, test_query, k=TOP_K_RESULTS)

# 👀 Inspect what the LLM will see
print(f"\n✅ Retrieved {len(retrieved_chunks)} chunks\n")
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"{'='*60}")
    print(f"[Chunk {i}] Page {chunk.metadata.get('page', 'N/A')}")
    print(f"{'='*60}")
    print(chunk.page_content)
    print()

print("💡 These are the exact chunks your LLM will use to answer the question.")
print("   Does this context contain what's needed for a good answer?")

### 6. LLM Configuration Parameters

| Parameter | Value | Purpose |
|-----------|-------|---------|
| **model_id** | `Qwen/Qwen2.5-7B-Instruct` | Which LLM to use via HuggingFace API |
| **temperature** | 0.3 | Controls randomness (0=deterministic, 1=creative) |
| **max_tokens** | 500 | Maximum answer length (prevents runaway responses) |

We're using HuggingFace's **Inference API**:
- Runs models on HuggingFace's servers
- No need to download huge models
- Simple API

### Prompt Engineering: The 3-Part Prompt Structure

```
[SYSTEM MESSAGE]
"You are a helpful assistant that answers questions based on provided documents.
Answer only based on the context provided. If the context does not contain the answer,
say 'I don't know.'"

[CONTEXT]
The retrieved chunks from our vector store
"Retrieved documents:
[Document 1]
...text from chunk 1...

[Document 2]
...text from chunk 2..."

[QUESTION]
The user's question
"Question: How does the perceptron algorithm learn?"
```

### Why This Structure?

1. **System Message**: Tells LLM what role to play and constraints
2. **Context**: Provides factual information to base answer on
3. **Question**: What we actually want answered

This ensures:
- ✅ LLM only uses provided context (no hallucination)
- ✅ Answers are grounded in your PDF
- ✅ Clear and consistent responses

---

#### ✏️ Exercise 3: Choose Your Model

| Model | Size | Character |
|---|---|---|
| `Qwen/Qwen2.5-7B-Instruct` | 7B | Strong instruction following |
| `meta-llama/Llama-3.1-8B-Instruct` | 8B | Well-rounded, widely used |
| `mistralai/Mistral-7B-Instruct-v0.3` | 7B | Fast and efficient |
| `google/gemma-2-9b-it` | 9B | Google's instruction-tuned model |
| `microsoft/Phi-3.5-mini-instruct` | 3.8B | Smallest and fastest |

Pick one. Your choice will affect answer quality, verbosity, and — later in Phase 3 — how well the model follows citation instructions.

In [ ]:
# ✏️ Exercise 3: Choose your model
# Uncomment one of the options below and copy it into HF_MODEL_ID:
# "Qwen/Qwen2.5-7B-Instruct"          # Strong instruction following
# "meta-llama/Llama-3.1-8B-Instruct"  # Well-rounded, widely used
# "mistralai/Mistral-7B-Instruct-v0.3" # Fast and efficient
# "google/gemma-2-9b-it"               # Google's instruction-tuned model
# "microsoft/Phi-3.5-mini-instruct"    # Smallest and fastest (3.8B)
HF_MODEL_ID = None  # TODO: replace with your chosen model ID (copy from above)

assert HF_MODEL_ID is not None, "Please set HF_MODEL_ID to a model string"

# --- The rest is fixed config, no changes needed below ---
LLM_TEMPERATURE = 0.3
LLM_MAX_TOKENS = 500

model_config = {
    'model_id': HF_MODEL_ID,
    'temperature': LLM_TEMPERATURE,
    'max_tokens': LLM_MAX_TOKENS,
}

SYSTEM_PROMPT = """You are a helpful assistant that answers questions based on provided documents.
Answer only based on the context provided. If the context does not contain the answer, say 'I don't know.'"""

def format_prompt(question: str, chunks: List[Document], system_prompt: str) -> str:
    """
    Format the prompt with system message, context chunks, and user question.

    Args:
        question: User's question
        chunks: List of Document objects from retrieval
        system_prompt: System instructions for the LLM

    Returns:
        Formatted prompt string with 3 parts: system | context | question
    """
    # Format context chunks
    context_text = "Retrieved documents:\n"
    for i, chunk in enumerate(chunks, 1):
        context_text += f"\n[Document {i}]\n{chunk.page_content}\n"

    # Combine all parts into a 3-part prompt
    prompt = f"""{system_prompt}

    {context_text}

    Question: {question}

    Answer:"""

    return prompt

def generate_answer(
    prompt: str,
    model_id: str,
    api_key: str,
    temperature: float = 0.3,
    max_tokens: int = 500
) -> str:
    """
    Call HuggingFace Inference API to generate an answer using InferenceClient.
    """
    client = InferenceClient(api_key=api_key)

    completion = client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return completion.choices[0].message.content

print("✅ LLM configuration loaded")

### 7. RAG Pipeline

Now we combine all the stages into a complete **Retrieval-Augmented Generation** pipeline:

```
User Question
    ↓
Step 1: Retrieve (Find relevant chunks from PDF)
    ↓
Step 2: Format Prompt (Combine system message + context + question)
    ↓
Step 3: Generate (Call LLM API to get answer)
    ↓
Chatbot Response + Sources
```

In [ ]:
def rag_pipeline(vector_store, user_question, system_prompt, model_config):
    """
    Complete RAG pipeline: Retrieve → Format → Generate

    Args:
        vector_store: FAISS vector store with embeddings
        user_question: User's question
        system_prompt: System instructions for the LLM
        model_config: Dictionary with model settings (model_id, temperature, etc.)

    Returns:
        Dictionary with answer, sources, and metadata
    """

    print("\n" + "="*60)
    print("RAG PIPELINE EXECUTION")
    print("="*60)

    try:
        # STAGE 1: RETRIEVAL
        print("\n[Step 1/3] Retrieving relevant chunks...")
        retrieved_chunks = retrieval(
            vector_store,
            user_question,
            k=TOP_K_RESULTS
        )
        print(f"✅ Retrieved {len(retrieved_chunks)} chunks")

        # STAGE 2: PROMPT FORMATTING
        print("\n[Step 2/3] Formatting prompt...")
        prompt = format_prompt(user_question, retrieved_chunks, system_prompt)
        print("✅ Prompt formatted")

        # STAGE 3: GENERATION
        print("\n[Step 3/3] Calling LLM API for answer generation...")
        answer = generate_answer(
            prompt=prompt,
            model_id=model_config['model_id'],
            api_key=HF_TOKEN,
            temperature=model_config['temperature'],
            max_tokens=model_config['max_tokens']
        )
        print("✅ Answer generated successfully")

        # COMPILE RESULTS
        result = {
            'question': user_question,
            'answer': answer,
            'sources': retrieved_chunks,
            'source_count': len(retrieved_chunks)
        }

        return result

    except Exception as e:
        print(f"\n❌ Error in RAG pipeline: {e}")
        raise

print("✅ RAG pipeline function loaded and ready")

### 8. Test the RAG Pipeline

Try different questions and see how well your RAG pipeline works!

In [ ]:
def display_result(result):
    """Display the RAG pipeline result in a readable format."""
    print("CHATBOT RESPONSE")

    print(f"\n Question: {result['question']}")

    print(f"\n Answer:\n")
    print(result['answer'])

    print(f"\n Sources ({result['source_count']} retrieved chunks):")
    print("-"*60)

    for i, source in enumerate(result['sources'], 1):
        print(f"\n[Source {i}] (Page {source.metadata.get('page', 'N/A')})")
        content_preview = source.page_content[:250] + "..." if len(source.page_content) > 250 else source.page_content
        print(content_preview)

question = "What are the key takeaways of the 2026 Budget?" # Change the question to whatever you'd like to ask.
result = rag_pipeline(vector_store, question, SYSTEM_PROMPT, model_config)
display_result(result)

## 2️⃣ Phase 2: Source Attribution

What we have done so far:
```
PDF → Load → Chunk → Embed → Store
Question → Retrieve → Format Prompt → LLM → Display
```

What we will be doing in Phase 2:
```
PDF → Load → Chunk → [ENRICH WITH CITATIONS] → Embed → Store

Question → Retrieve → Format Prompt [WITH CITATION INSTRUCTIONS] →
LLM generates → Extract Citations → Link to Metadata → Format Output
```

### 1. Citation Data Structures

#### BoundingBox: PDF Coordinates
Each citation needs to know **where** in the PDF it came from (for click-to-jump in future UIs):
- **x0, y0**: Top-left corner of text
- **x1, y1**: Bottom-right corner of text
- Measured in PDF points (72 points = 1 inch)

#### Citation: Complete Source Information
Every chunk gets enriched with:
- **citation_id**: Unique ID like `c3.0.0` (page 3, section 0, chunk 0)
- **page_number**: Which page the text came from
- **paragraph_id**: Human-readable location (e.g., "P3-Para0")
- **section_title**: Extracted heading
- **bbox**: Coordinates for click-to-jump
- **text_preview**: First 100 chars for quick reference

In [ ]:
@dataclass
class BoundingBox:
    """Represents spatial coordinates of text in a document."""
    x0: float
    y0: float
    x1: float
    y1: float

    def to_dict(self) -> Dict:
        return asdict(self)

@dataclass
class Citation:
    """Represents a single citation location in a document."""
    citation_id: str
    page_number: int
    paragraph_id: str
    section_title: Optional[str] = None
    bbox: Optional[BoundingBox] = None
    text_preview: str = ""

    def to_dict(self) -> Dict:
        return {
            "citation_id": self.citation_id,
            "page_number": self.page_number,
            "paragraph_id": self.paragraph_id,
            "section_title": self.section_title,
            "bbox": self.bbox.to_dict() if self.bbox else None,
            "text_preview": self.text_preview,
        }

### 2. CitationExtractor: Enriching Chunks with Metadata

#### What It Does
During indexing (Phase 2 of RAG), we enrich each chunk with citation information:
- **Extract page number** from metadata
- **Generate citation ID** like `c3.0.0`
- **Extract section title** from text
- **Calculate bounding box** using pdfplumber (or estimate)

#### Key Methods
- **`_generate_citation_id()`**: Creates unique ID for each chunk
- **`_extract_page_number()`**: Gets page from metadata
- **`_extract_section_title()`**: Finds heading in chunk
- **`_estimate_bbox()`**: Gets PDF coordinates (accurate via pdfplumber, or estimated)
- **`enrich_chunk()`**: Main function - adds all metadata to a chunk

In [ ]:
try:
    import pdfplumber
    PDFPLUMBER_AVAILABLE = True
except ImportError:
    PDFPLUMBER_AVAILABLE = False

class CitationExtractor:
    """
    Extracts and enriches document chunks with citation metadata.
    Converts standard Document objects into citation-aware chunks.
    """

    def __init__(self, pdf_path: Optional[str] = None):
        self.citation_counter = 0
        self.pdf_path = pdf_path
        self.pdf = None
        if pdf_path and PDFPLUMBER_AVAILABLE:
            try:
                self.pdf = pdfplumber.open(pdf_path)
            except Exception as e:
                print(f"⚠️  Could not open PDF with pdfplumber: {e}")
                self.pdf = None

    def _generate_citation_id(self, page: int, para_idx: int, section_idx: int = 0) -> str:
        """Generate standardized citation ID: P{page}-S{section}-P{paragraph}"""
        return f"c{page}.{section_idx}.{para_idx}"

    def _extract_page_number(self, document: Document) -> int:
        """Extract page number from document metadata."""
        if hasattr(document, 'metadata') and document.metadata:
            return document.metadata.get('page', 0)
        return 0

    def _extract_section_title(self, text: str, max_chars: int = 100) -> str:
        """Try to extract section title from text start."""
        lines = text.split('\n')
        for line in lines[:3]:
            stripped = line.strip()
            if 3 < len(stripped) < max_chars and not stripped.startswith('#'):
                return stripped
        return ""

    def _estimate_bbox(self, text_length: int, page: int) -> BoundingBox:
        """
        Estimate bounding box based on text length.
        Uses pdfplumber for actual coordinates if PDF is available,
        otherwise falls back to estimation.
        """
        # Try to get actual bounding box from PDF
        if self.pdf and page > 0 and page <= len(self.pdf.pages):
            try:
                pdf_page = self.pdf.pages[page - 1]  # Convert to 0-indexed

                # Get page dimensions
                page_width = pdf_page.width
                page_height = pdf_page.height

                # Extract text with bounding boxes
                text_objects = pdf_page.chars

                if text_objects:
                    # Get the bounding boxes of first N characters for this chunk
                    char_count = 0
                    bbox_info = []

                    for char in text_objects:
                        bbox_info.append({
                            'x0': char['x0'],
                            'y0': char['y0'],
                            'x1': char['x1'],
                            'y1': char['y1']
                        })
                        char_count += 1
                        if char_count >= text_length:
                            break

                    if bbox_info:
                        # Calculate bounding box from all characters
                        all_x0 = [b['x0'] for b in bbox_info]
                        all_y0 = [b['y0'] for b in bbox_info]
                        all_x1 = [b['x1'] for b in bbox_info]
                        all_y1 = [b['y1'] for b in bbox_info]

                        return BoundingBox(
                            x0=min(all_x0),
                            y0=min(all_y0),
                            x1=max(all_x1),
                            y1=max(all_y1)
                        )
            except Exception as e:
                # Fall back to estimation if anything goes wrong
                pass

        # Fallback: Estimate based on text length
        # Typical letter: 8.5" x 11" = 612 x 792 points
        avg_chars_per_line = 80
        line_height = 12  # points
        lines = max(1, text_length // avg_chars_per_line)

        # Position based on page progression
        y_pos = 50 + ((page % 5) * 150)  # Distribute across page

        return BoundingBox(
            x0=50,
            y0=y_pos,
            x1=562,  # 612 - 50 margin
            y1=y_pos + (lines * line_height)
        )

    def enrich_chunk(
        self,
        document: Document,
        chunk_index: int,
        section_title: Optional[str] = None
    ) -> Document:
        """
        Enrich a document chunk with citation metadata.

        Args:
            document: LangChain Document to enrich
            chunk_index: Index of chunk within the section
            section_title: Optional section title for context

        Returns:
            Enhanced Document with citation metadata
        """
        page_num = self._extract_page_number(document)

        # Generate citation ID
        citation_id = self._generate_citation_id(page_num, chunk_index)

        # Extract or use provided section title
        section = section_title or self._extract_section_title(document.page_content)

        # Estimate bounding box
        bbox = self._estimate_bbox(len(document.page_content), page_num)

        # Create citation object
        citation = Citation(
            citation_id=citation_id,
            page_number=page_num,
            paragraph_id=f"P{page_num}-Para{chunk_index}",
            section_title=section,
            bbox=bbox,
            text_preview=document.page_content[:100]
        )

        # Update metadata with citation info
        if not document.metadata:
            document.metadata = {}

        document.metadata.update({
            "citation_id": citation_id,
            "page_number": page_num,
            "paragraph_id": citation.paragraph_id,
            "section_title": section,
            "citation_bbox": bbox.to_dict(),
            "citation_data": citation.to_dict()
        })

        return document


#### Helper Function
Wrapper function that applies `CitationExtractor` to all chunks at once.

In [ ]:
def create_citation_aware_chunks(
    documents: List[Document],
    pdf_path: Optional[str] = None,
    enrich: bool = True,
) -> List[Document]:
    """
    Convert a list of documents into citation-aware chunks.

    Args:
        documents: List of Document objects from retrieval
        pdf_path: Optional path to PDF for accurate bounding boxes
        enrich: Whether to enrich with citation metadata
    Returns:
        List of enriched documents with citation data
    """
    extractor = CitationExtractor(pdf_path)
    citation_docs = []

    for chunk_idx, doc in enumerate(documents):
        if enrich:
            doc = extractor.enrich_chunk(doc, chunk_idx)
        citation_docs.append(doc)

    return citation_docs


### 3. Citation-Aware Chunking

In Phase 1, we chunked the PDF normally. Now we:
1. **Chunk** the PDF (same as Phase 1)
2. **Enrich** each chunk with citation metadata (NEW)

This modified version calls `create_citation_aware_chunks()` with the PDF path to extract bounding boxes.

In [ ]:
def chunk_documents_citation(documents, chunk_size=500, chunk_overlap=100, pdf_path=None):
    """Split documents into smaller chunks with optional citation metadata."""

    print(f"Splitting into chunks (size={chunk_size}, overlap={chunk_overlap})")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    chunks = text_splitter.split_documents(documents)
    print(f"✅ Created {len(chunks)} chunks")


    print("Adding citation metadata to chunks...")
    chunks = create_citation_aware_chunks(chunks, pdf_path=pdf_path, enrich=True)
    print(f"✅ Enhanced {len(chunks)} chunks with citations")

    return chunks

#### Helper Function
Extract and display citation information from a single chunk. Useful for debugging and understanding what metadata each chunk has.

In [ ]:
def get_chunk_citations(chunk: Document) -> dict:
    """
    Extract citation information from an enriched chunk.

    Args:
        chunk: Document with citation metadata

    Returns:
        Dictionary with citation details
    """
    metadata = {
        "citation_id": chunk.metadata.get("citation_id", "unknown"),
        "page": chunk.metadata.get("page_number", 0),
        "paragraph_id": chunk.metadata.get("paragraph_id", ""),
        "section": chunk.metadata.get("section_title", ""),
        "bbox": chunk.metadata.get("citation_bbox"),
        "preview": chunk.page_content[:150]
    }
    print("="*60)
    print(f"Citation ID: {metadata['citation_id']}")
    print(f"Page: {metadata['page']}")
    print(f"Paragraph ID: {metadata['paragraph_id']}")
    print(f"Section: {metadata['section']}")
    print(f"Preview: {metadata['preview']}\n")
    print(f"Bounding Box: {metadata['bbox']}")
    print("="*60)


#### Try Out Chunking!

In [ ]:
documents = load_pdf(PDF_PATH)
chunks = chunk_documents_citation(
            documents,
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            pdf_path=PDF_PATH
        )

# Inspect what we got
print("\n--- Sample Chunk---")
print(f"Total chunks: {len(chunks)}")

# TO DO: use get chunk_citations to preview a citation by printing


### 4. Citation-Aware Retrieval

#### Change from Phase 1
In Phase 1, we retrieve chunks but lose citation metadata during retrieval. Now we:
1. **Retrieve** chunks (same as Phase 1)
2. **Check metadata** - if chunk already has citation_id, use it
3. **Re-enrich** - if chunk missing citations, extract them now
4. **Return citation-aware chunks** for next stages

This ensures all retrieved chunks have citation information ready for the LLM.

In [ ]:
def retrieval_with_citations(
    vector_store,
    query: str,
    k: int = 3,
) -> List[Document]:
    """
    Advanced retrieval that returns citation-aware chunks.

    Args:
        vector_store: FAISS vector store
        query: Search query
        k: Number of results to retrieve

    Returns:
        List of Documents with citation metadata and markers
    """
    print(f"\n[CITATION-AWARE RETRIEVAL]")
    print(f"Query: '{query}'")
    print(f"Retrieving top {k} chunks with citations...\n")

    results = vector_store.similarity_search(query, k=k)

    # Ensure all chunks have citation metadata
    citation_chunks = []
    extractor = CitationExtractor()

    for idx, chunk in enumerate(results):
        # Check if already enriched
        if "citation_id" not in chunk.metadata:
            chunk = extractor.enrich_chunk(chunk, idx)


        citation_chunks.append(chunk)

    print(f"✅ Retrieved {len(citation_chunks)} citations\n")
    return citation_chunks

#### Try It Out Here!

In [ ]:
vector_store = create_vector_store(chunks, EMBEDDING_MODEL)
save_vector_store(vector_store, save_path=FAISS_INDEX_PATH)

# Test retrieval with a sample query
print("TESTING RETRIEVAL")
test_query = #FIX ME (e.g., "What did PM Wong say about AI adoption?")

print("="*60)
retrieved_chunks = retrieval_with_citations(vector_store, test_query, k=TOP_K_RESULTS)

for chunk in retrieved_chunks:
    metadata = get_chunk_citations(chunk)


### 5. Citation-Aware Prompt Formatting

#### How Prompt Changes
In Phase 1, we just showed the retrieved documents. In Phase 2, we:
1. **Add citation guidelines** - Tell LLM to cite sources with `[c3.0.0]` format
2. **Show metadata** - Include citation IDs, page numbers, sections in context
3. **Give examples** - Show format like `[c3.0.0]` and `[c3.0.0, c5.0.1]`
4. **Request citations** - Ask for "comprehensive answer that includes specific citations"

#### Prompt Engineering Goal
The LLM now sees citation IDs in the context and knows it should:
- Use `[c3.0.0]` format for inline citations
- Place citations immediately after claims
- Use ONLY the citation IDs provided (prevents hallucination)

In [ ]:
# Enhanced RAG LLM Prompt Formatting i.e. w Citations
def format_citation_aware_prompt(
    question: str,
    chunks: List[Document],
    system_prompt: str,
    include_instructions: bool = True
) -> str:
    """
    Format a prompt that encourages the LLM to reference source citations.

    Returns:
        Enhanced prompt with citation guidance
    """
    # Build enhanced system prompt
    enhanced_system = system_prompt
    if include_instructions:
        enhanced_system += """

Use citations in your answer to reference the sources used to answer the question.

IMPORTANT - Citation Guidelines:
- When making a claim, immediately cite the source using its Citation ID in square brackets
- Place citations inline right after the relevant text: e.g. "Government will support workers through skill acquisition and role adaptation [c28.0.79]."
- If a claim uses multiple sources, list all citation IDs: e.g. "PM Wong views AI as a strategic asset to overcome structural constraints [c22.0.60, c22.0.61]."

# TO DO: Add more guidelines in prompt

"""

    # Format context with citation metadata
    context_text = "Retrieved documents with citations:\n"
    for i, chunk in enumerate(chunks, 1):
        citation_id = chunk.metadata.get("citation_id", f"Doc{i}")
        page = chunk.metadata.get("page_number", "?")
        section = chunk.metadata.get("section_title", "Untitled")
        paragraph_id = chunk.metadata.get("paragraph_id", "")

        context_text += f"\n{'='*60}\n"
        context_text += f"[Document {i}]\n"
        context_text += f"Citation ID: {citation_id}\n"
        context_text += f"Page: {page}\n"
        context_text += f"Section: {section}\n"
        context_text += f"Paragraph ID: {paragraph_id}\n"
        context_text += f"{'='*60}\n"
        context_text += f"{chunk.page_content}\n"

    prompt = f"""{enhanced_system}

{context_text}

Question: {question}

Provide a comprehensive answer that includes specific citations to the documents above.
Answer:"""

    return prompt

### 6. Citation Formatter

#### What It Does
After the LLM generates an answer with citations like `[c3.0.0]`, we need to **format it for display**:
- Keep the answer text
- Add a "SOURCES" section below
- For each citation ID found, show:
  - Page number
  - Section title
  - Full content

This gives users a readable reference list of exactly which sources were cited.

In [ ]:
def citation_formatter(
    response: str,
    citations: List[Dict],
    citation_lookup: Dict
) -> str:
    """Convert to plain text with numbered citations."""
    text = f"{response}\n\n"
    text += "=" * 60 + "\n"
    text += f"SOURCES ({len(citations)} cited):\n"
    text += "=" * 60 + "\n\n"

    for i, citation in enumerate(citations, 1):
        cite_id = citation["id"]
        source = citation["source"]

        text += f"[{i}] Page {source['page']} | {source.get('section', 'Untitled')}\n"
        text += f"    ID: {cite_id}\n"
        text += f"    Content: {source.get('full_content', source['preview'])}\n\n"

    return text

### 7. CitationLinker: The Post-Processing Engine

The LLM generates text like: `"PM Wong frames AI as a strategic advantage [c22.0.60] that must be deployed with discipline to address Singapore's structural constraints."`

But we need to:
- Extract citation IDs `c22.0.60` using regex
- Look up their source information in our retrieved chunks
- Create a structured mapping of citations to sources

#### Key Methods
- **`extract_citations_from_response()`**: Regex to find all `[c22.0.60]` patterns
- **`link_citations()`**: Main function - builds lookup table and links citations

In [ ]:
import re

class CitationLinker:
    """
    Post-processes LLM responses to convert citation markers into clickable links.
    Creates structured output with citation metadata.
    """

    def __init__(self):
        # Updated pattern to match [c3.0.0] or [c3.0.0, c5.0.1]
        self.citation_pattern = re.compile(r'\[(c\d+\.\d+\.\d+(?:,\s*c\d+\.\d+\.\d+)*)\]')

    def extract_citations_from_response(self, text: str) -> List[str]:
        """Extract all citation IDs from response text."""
        matches = self.citation_pattern.findall(text)

        # Flatten matches (handle both single and multiple citations)
        citation_ids = []
        for match in matches:
            # Split by comma in case of multiple IDs: "c3.0.0, c5.0.1"
            ids = [cid.strip() for cid in match.split(',')]
            citation_ids.extend(ids)

        return list(set(citation_ids))  # Remove duplicates

    def link_citations(
        self,
        response: str,
        retrieved_chunks: List[Document]
    ) -> Dict[str, Any]:
        """
        Post-process response to link citations with source documents.

        Args:
            response: LLM-generated response text
            retrieved_chunks: Retrieved documents with citation metadata

        Returns:
            Dictionary with linked response and citation metadata
        """
        # Build citation lookup from chunks
        citation_lookup = {}
        for chunk in retrieved_chunks:
            citation_id = chunk.metadata.get("citation_id")
            if citation_id:
                citation_lookup[citation_id] = {
                    "page": chunk.metadata.get("page_number"),
                    "paragraph_id": chunk.metadata.get("paragraph_id"),
                    "section": chunk.metadata.get("section_title"),
                    "bbox": chunk.metadata.get("citation_bbox"),
                    "preview": chunk.page_content[:200],
                    "full_content": chunk.page_content  # Full content for citations
                }

        # Extract citations from response
        cited_ids = self.extract_citations_from_response(response)

        # Build citation links
        citations = []
        for cid in cited_ids:
            if cid in citation_lookup:
                citations.append({
                    "id": cid,
                    "source": citation_lookup[cid]
                })

        # Clean response (citation markers are kept in square brackets - no removal needed)
        clean_response = response

        return {
            "response": clean_response,
            "citations": citations,
            "citation_count": len(citations),
            "sources": citation_lookup
        }


### 8. Generate Answer with Citations
```
Citation-aware Prompt
    ↓
LLM generates: "...answer [c3.0.0]..."
    ↓
CitationLinker extracts: "c3.0.0"
    ↓
Link to sources: {page: 3, section: "...", bbox: {...}}
    ↓
citation_formatter creates display text
    ↓
Return structured result
```

In [ ]:
def generate_answer_with_citations(
    prompt: str,
    model_id: str,
    api_key: str,
    retrieved_chunks: List[Document],
    temperature: float = 0.3,
    max_tokens: int = 500,
    output_format: str = "json"
) -> dict:
    """
    Generate an answer and post-process it with citation links.

    Returns:
        Dictionary with response and citation metadata
    """
    # Generate answer from LLM
    response = generate_answer(
        prompt=prompt,
        model_id=model_id,
        api_key=api_key,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    # Post-process with citations
    linker = CitationLinker()
    linked_response = linker.link_citations(response, retrieved_chunks)

    # Format output

    formatted = citation_formatter(
        linked_response["response"],
        linked_response["citations"],
        linked_response["sources"]
    )

    return {
        "raw_response": response,
        "linked_response": linked_response,
        "formatted": formatted,
        "output_format": output_format
    }


### 9. Complete Citation-Aware RAG Pipeline
Now we combine all the pieces for a complete pipeline with citations:

```
User Question
    ↓
Step 1: retrieval_with_citations() → Retrieve chunks + metadata
    ↓
Step 2: format_citation_aware_prompt() → Create prompt with guidelines
    ↓
Step 3: generate_answer_with_citations() → LLM + extract + link + format
    ↓
Final Output: Answer with citations + formatted sources
```

In [ ]:
def rag_pipeline_citation(vector_store, user_question, system_prompt):
    """
    Complete RAG pipeline: Retrieve → Format → Generate

    Args:
        vector_store: FAISS vector store with embeddings
        user_question: User's question
        system_prompt: System instructions for the LLM
        model_config: Dictionary with model settings (model_id, temperature, etc.)

    Returns:
        Dictionary with answer, sources, and metadata
    """

    print("\n" + "="*60)
    print("RAG PIPELINE EXECUTION")
    print("="*60)

    try:
        # STAGE 1: RETRIEVAL
        print("\n[Step 1/3] Retrieving relevant chunks...")
        retrieved_chunks = retrieval_with_citations(
            vector_store,
            user_question,
            k=TOP_K_RESULTS
        )
        print(f"✅ Retrieved {len(retrieved_chunks)} chunks")

        # STAGE 2: PROMPT FORMATTING
        print("\n[Step 2/3] Formatting prompt...")
        prompt = format_citation_aware_prompt(user_question, retrieved_chunks, system_prompt)
        print("✅ Prompt formatted")

        # STAGE 3: GENERATION
        print("\n[Step 3/3] Calling LLM API for answer generation...")

        result = generate_answer_with_citations(
                prompt=prompt,
                model_id=HF_MODEL_ID,
                api_key=HF_TOKEN,
                retrieved_chunks=retrieved_chunks,
                temperature=LLM_TEMPERATURE,
                max_tokens=LLM_MAX_TOKENS
            )
        print("✅ Answer generated successfully")

        # COMPILE RESULTS
        output = {
            'question': user_question,
            'answer': result["linked_response"]["response"],
            "citations": result["linked_response"]["citations"],
            'sources': retrieved_chunks,
            'formatted': result["formatted"],
            'source_count': len(retrieved_chunks)
        }

        return output

    except Exception as e:
        print(f"\n❌ Error in RAG pipeline: {e}")
        raise

print("✅ RAG pipeline function loaded and ready")

### Complete Phase 2

Run a question through the complete Phase 2 pipeline and see:
- Answer with inline citations: `[c3.0.0]`, `[c5.0.1]`
- Formatted sources list showing page numbers, sections, and full content
- Structured citation data linking answer text to source chunks

In [ ]:
question = #FIXME (e.g., "What is the budget for 2026?")
result = rag_pipeline_citation(vector_store, question, SYSTEM_PROMPT)

print("CHATBOT RESPONSE")
print(f"Question: \n {result['question']}\n")
print(f"Answer: \n")
print(result['formatted'])

## 3️⃣ Phase 3: LLM As-A-Judge

What we have done so far:
```
PDF → Load → Chunk → [ENRICH WITH CITATIONS] → Embed → Store

Question → Retrieve → Format Prompt [WITH CITATION INSTRUCTIONS] →
LLM generates → Extract Citations → Link to Metadata → Format Output
```

What we will be doing in Phase 3:
```
PDF → Load → Chunk → [ENRICH WITH CITATIONS] → Embed → Store

Question → Retrieve → Format Prompt [WITH CITATION INSTRUCTIONS] →
LLM generates → Extract Citations → Link to Metadata →
[EVALUATE CLAIMS AGAINST SOURCES] → Format Output with Confidence Scores
```

### The Problem with Phase 2

In Phase 2, our chatbot now produces answers with inline citations like `[c3.0.0]`. But there's a critical gap: **we never actually verify those citations**.

The LLM might:
- Cite a source that **doesn't support** the claim it's making
- **Misrepresent** what the source says
- **Hallucinate** a claim and attach a real citation to it

#### The Solution: LLM-as-a-Judge

We use a **second LLM call** to evaluate each cited claim against its actual source text. This judge LLM doesn't generate answers — it reads claim + source and produces a structured verdict:

| Support Level | Meaning |
|---------------|---------|
| `supported` | The source clearly backs the claim |
| `partial` | The source partially supports the claim |
| `unsupported` | The source does not mention or support the claim |
| `contradicted` | The source says the opposite |
| `uncited` | No citation was given (no source to check) |

Each verdict comes with a **confidence score** (0.0–1.0) and a **reasoning** sentence. The overall confidence of the response is the average score across all **cited** claims.

### 1. Evaluation Data Structures

We need two dataclasses to hold the evaluation results.

#### ClaimEvaluation — One evaluated claim
| Field | Type | Description |
|-------|------|-------------|
| `claim_text` | `str` | The sentence being evaluated (citation markers removed) |
| `citation_ids` | `List[str]` | Which sources were cited (e.g. `["c1.0.2"]`) |
| `support_level` | `str` | One of: `supported`, `partial`, `unsupported`, `contradicted`, `uncited` |
| `confidence` | `float` | Score 0.0–1.0 assigned by the evaluator |
| `reasoning` | `str` | One-sentence explanation of the verdict |

#### EvaluationResult — The full evaluation of a response
Aggregates all `ClaimEvaluation` objects and computes summary statistics.

- `overall_confidence`: Mean confidence of **cited claims only** (uncited claims are excluded from scoring — they are noted but not penalised)
- `total_claims`, `supported_count`, `partial_count`, `unsupported_count`, `contradicted_count`, `uncited_count`: Counts for each support level

In [ ]:
import json

@dataclass
class ClaimEvaluation:
    """Result of evaluating a single claim against its cited source."""
    claim_text: str
    citation_ids: List[str]
    support_level: str   # "supported" | "partial" | "unsupported" | "contradicted" | "uncited"
    confidence: float    # 0.0–1.0
    reasoning: str

@dataclass
class EvaluationResult:
    """Complete evaluation of an LLM response."""
    overall_confidence: float
    claim_evaluations: List[ClaimEvaluation]
    total_claims: int
    supported_count: int
    partial_count: int
    unsupported_count: int
    contradicted_count: int
    uncited_count: int
    evaluator_model: str

    def to_dict(self) -> Dict[str, Any]:
        return {
            "overall_confidence": self.overall_confidence,
            "total_claims": self.total_claims,
            "supported": self.supported_count,
            "partial": self.partial_count,
            "unsupported": self.unsupported_count,
            "contradicted": self.contradicted_count,
            "uncited": self.uncited_count,
            "evaluator_model": self.evaluator_model,
            "claims": [asdict(c) for c in self.claim_evaluations],
        }

print("✅ Evaluation data structures loaded")

### 2. Claim Extraction

Before we can evaluate anything, we need to **split the LLM's response into individual claims**.

#### Strategy: Sentence-Level Splitting
We use a regex to split on sentence boundaries (`.`, `!`, `?`). Each sentence is one "claim".

For each sentence we:
1. Extract any `[c3.0.0]` citation markers using the same regex pattern from Phase 2
2. Strip the markers from the claim text (so the evaluator reads clean prose)
3. Clean up any stray whitespace left behind (e.g. `"claim ."` → `"claim."`)
4. Store `{"text": "...", "citation_ids": ["c3.0.0"]}`

Sentences with **no citations** get `citation_ids: []`. They'll be labelled `uncited` later — we can't verify what has no source.

Very short fragments (< 10 chars) are merged with the previous sentence so we don't send meaningless one-word "claims" to the evaluator.

_Note: This might not be the best way of defining claims, but it’s something you can try to improve on after this workshop. For example, you could try splitting a response by clauses instead._

In [ ]:
# Same citation pattern used by CitationLinker in Phase 2
CITATION_PATTERN = re.compile(r'\[(c\d+\.\d+\.\d+(?:,\s*c\d+\.\d+\.\d+)*)\]')


def _extract_citation_ids(text: str) -> List[str]:
    """Extract all citation IDs from a text string, preserving order and deduplicating."""
    matches = CITATION_PATTERN.findall(text)
    citation_ids = []
    for match in matches:
        ids = [cid.strip() for cid in match.split(',')]
        citation_ids.extend(ids)
    # Deduplicate while preserving order
    seen = set()
    unique = []
    for cid in citation_ids:
        if cid not in seen:
            seen.add(cid)
            unique.append(cid)
    return unique


def extract_claims(response_text: str) -> List[Dict[str, Any]]:
    """
    Split an LLM response into individual claims with associated citation IDs.

    Args:
        response_text: The full LLM response string with [cX.Y.Z] markers

    Returns:
        List of dicts: [{"text": str, "citation_ids": List[str]}, ...]
    """
    # Split on sentence boundaries, preserving the delimiter
    sentences = re.split(r'(?<=[.!?])\s+', response_text.strip())

    claims = []
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        # Merge very short fragments with the previous claim
        if len(sentence) < 10 and claims:
            claims[-1]["text"] += " " + sentence
            merged_ids = _extract_citation_ids(claims[-1]["text"])
            claims[-1]["citation_ids"] = merged_ids
            claims[-1]["text"] = CITATION_PATTERN.sub('', claims[-1]["text"]).strip()
            continue

        citation_ids = _extract_citation_ids(sentence)

        # Remove citation markers from the claim text for cleaner evaluation
        clean_text = CITATION_PATTERN.sub('', sentence).strip()
        # Remove stray whitespace before punctuation (e.g. "claim ." → "claim.")
        clean_text = re.sub(r'\s+([.!?,;:])', r'\1', clean_text)

        if clean_text:
            claims.append({"text": clean_text, "citation_ids": citation_ids})

    return claims

print("✅ Claim extraction functions loaded")

#### Try it out: inspect extracted claims

Run the cell below to see what `extract_claims()` produces on a sample response.

In [ ]:
sample_response = (
    "The perceptron is a linear classifier [c1.0.0]. It learns by adjusting weights whenever it makes a mistake [c1.0.1, c2.0.0]. The algorithm always converges when the data is linearly separable [c2.0.1]. This makes it a foundational model in machine learning."
)

claims = extract_claims(sample_response)

print(f"Extracted {len(claims)} claims:\n")
for i, claim in enumerate(claims, 1):
    status = f"cited: {claim['citation_ids']}" if claim['citation_ids'] else "UNCITED"
    print(f"[{i}] {claim['text']}")
    print(f"     → {status}")
    print()

### 3. Evaluation Prompt Engineering

We now have a list of claims and their citation IDs. The next step is to ask the evaluator LLM to score each one.

#### Two-message structure
Just like in Phase 1, we use a **system message** and a **user message**:

- **System message**: A strict instruction telling the LLM to respond **only** with a valid JSON array. No preamble, no explanation. This is critical — if the model adds "Sure! Here are the results..." before the JSON, our parser fails.
- **User message**: The list of claims, each paired with its source text (truncated to 500 chars to stay within token limits).

#### Expected output format
The evaluator should return a JSON array like this:

```json
[
  {
    "claim_index": 1,
    "support_level": "supported",
    "confidence": 0.95,
    "reasoning": "The source explicitly defines perceptron as a linear classifier."
  },
  {
    "claim_index": 2,
    "support_level": "partial",
    "confidence": 0.6,
    "reasoning": "The source mentions weight updates but does not describe the exact mechanism."
  }
]
```

We use **temperature = 0.1** (near-deterministic) for consistent, structured output.

### Try it Out: Evaluation System Prompt Modification

The evaluation system prompt controls how strict the fact-checker is. Try modifying it below:

**Current behavior:**
- Accepts citations as "supported" or "partial" if the source generally aligns
- Allows some interpretation flexibility

**How to make it stricter:**
- Add instructions like: `"Only mark as 'supported' if the source explicitly states the claim word-for-word"`
- Reduce allowed interpretation

**How to make it looser:**
- Add instructions like: `"Mark as 'supported' if the source discusses related concepts, even if not exact match"`

Modify `EVALUATION_SYSTEM_PROMPT` and run the cell below to see how it changes fact-checking behavior.

In [ ]:
EVALUATION_SYSTEM_PROMPT = """You are a fact-checking assistant. Your job is to evaluate whether claims are supported by provided source texts.

For each claim you receive, output a JSON object with these exact fields:
- "claim_index": the integer index of the claim (as given)
- "support_level": one of exactly "supported", "partial", "unsupported", or "contradicted"
- "confidence": a float between 0.0 and 1.0
- "reasoning": one or two sentences explaining your decision

Output ONLY a valid JSON array of these objects. Do not include any explanation, markdown, or other text outside the JSON array."""


def build_evaluation_prompt(
    claims: List[Dict[str, Any]],
    citation_lookup: Dict[str, Dict]
) -> str:
    """
    Build the user-turn prompt for the evaluator LLM.

    Only includes cited claims (uncited claims are skipped).
    The system instructions are kept separate in EVALUATION_SYSTEM_PROMPT.

    Args:
        claims: List from extract_claims() - each has "text" and "citation_ids"
        citation_lookup: Sources dict from link_citations(), keyed by citation_id,
                         each value has "full_content"

    Returns:
        Formatted user prompt string
    """
    prompt = "Evaluate the following claims against their sources:\n"

    claim_index = 0
    for claim in claims:
        if not claim["citation_ids"]:
            continue  # Skip uncited claims

        claim_index += 1
        prompt += f"\n---\nClaim {claim_index}: {claim['text']}\n"

        # Gather source content for all citations referenced by this claim
        for cid in claim["citation_ids"]:
            if cid in citation_lookup:
                source = citation_lookup[cid]
                content = source.get("full_content", source.get("preview", ""))
                # Truncate to 500 chars to stay within context limits
                if len(content) > 500:
                    content = content[:500] + "..."
                prompt += f"Source ({cid}): {content}\n"

    prompt += f"\nReturn a JSON array with {claim_index} objects, one per claim."
    return prompt

print("✅ Evaluation prompt functions loaded")

### 4. LLM-as-a-Judge Evaluation

Now we call the LLM with our carefully structured prompt and parse the JSON response.

#### Design: Single Batched Call
We send **all claims in one LLM call** rather than one call per claim. This avoids multiplying API latency by the number of claims (typically 3–8 sentences).

#### JSON Parsing with Fallbacks
LLMs don't always produce perfectly formatted JSON — they may:
- Add a preamble sentence before the `[...]`
- Wrap the array in a dict: `{"results": [...]}`

We handle this with a **3-tier fallback**:
1. **Tier 1**: Try `json.loads()` directly — succeeds most of the time
2. **Tier 2**: Regex to extract `[...]` from anywhere in the response
3. **Tier 3**: Return `[]` and let the positional fallback assign default scores

#### Claim Matching with 3 Strategies
After parsing, we match each evaluated item back to its original claim using:
1. **1-based `claim_index`** — the most common convention
2. **0-based `claim_index`** — some models use this
3. **Positional fallback** — use the i-th result in order regardless of index

#### Overall Confidence
`overall_confidence` is the **mean confidence of cited claims only**. Uncited claims are excluded from the score — they're noted in the output, but we have no source to verify them against, so penalising them would be unfair.

In [ ]:
def _parse_evaluation_response(raw: str) -> List[Dict]:
    """
    Parse evaluator LLM response as JSON with fallbacks.

    Tier 1: Direct json.loads()
    Tier 2: Regex extract JSON array from anywhere in the response
    Tier 3: Return empty list (positional fallback handles per-claim defaults)
    """
    # Tier 1: Try direct parse
    try:
        parsed = json.loads(raw.strip())
        if isinstance(parsed, list):
            return parsed
        # Unwrap if the model returned a dict containing the array
        if isinstance(parsed, dict):
            for v in parsed.values():
                if isinstance(v, list):
                    return v
    except (json.JSONDecodeError, ValueError):
        pass

    # Tier 2: Try to extract a JSON array with regex
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            if isinstance(parsed, list):
                return parsed
        except (json.JSONDecodeError, ValueError):
            pass

    # Tier 3: Give up — positional fallback will assign default scores
    print("  ⚠️  Could not parse evaluator response as JSON. Using default scores.")
    return []


def _build_result(evaluations: List[ClaimEvaluation], model_id: str) -> EvaluationResult:
    """Build an EvaluationResult from a list of ClaimEvaluations."""
    supported    = sum(1 for e in evaluations if e.support_level == "supported")
    partial      = sum(1 for e in evaluations if e.support_level == "partial")
    unsupported  = sum(1 for e in evaluations if e.support_level == "unsupported")
    contradicted = sum(1 for e in evaluations if e.support_level == "contradicted")
    uncited      = sum(1 for e in evaluations if e.support_level == "uncited")

    # Overall confidence: mean of CITED claims only (uncited excluded)
    cited_evals = [e for e in evaluations if e.support_level != "uncited"]
    overall = sum(e.confidence for e in cited_evals) / len(cited_evals) if cited_evals else 0.0

    return EvaluationResult(
        overall_confidence=round(overall, 2),
        claim_evaluations=evaluations,
        total_claims=len(evaluations),
        supported_count=supported,
        partial_count=partial,
        unsupported_count=unsupported,
        contradicted_count=contradicted,
        uncited_count=uncited,
        evaluator_model=model_id,
    )


def evaluate_claims(
    claims: List[Dict[str, Any]],
    citation_lookup: Dict[str, Dict],
    model_id: str,
    api_key: str,
    temperature: float = 0.1,
    max_tokens: int = 1000,
) -> EvaluationResult:
    """
    Send claims to the evaluator LLM and parse structured results.

    Args:
        claims: Output of extract_claims()
        citation_lookup: Sources dict with full_content (from link_citations())
        model_id: HuggingFace model ID for evaluation
        api_key: HF API key
        temperature: Low temperature for deterministic evaluation
        max_tokens: Max tokens for evaluation response

    Returns:
        EvaluationResult with per-claim scores and overall confidence
    """
    cited_claims   = [c for c in claims if c["citation_ids"]]
    uncited_claims = [c for c in claims if not c["citation_ids"]]

    # Build evaluations list, starting with uncited (no LLM call needed for these)
    evaluations = []
    for claim in uncited_claims:
        evaluations.append(ClaimEvaluation(
            claim_text=claim["text"],
            citation_ids=[],
            support_level="uncited",
            confidence=0.0,
            reasoning="No citation provided for this claim.",
        ))

    # If there are no cited claims, return early
    if not cited_claims:
        return _build_result(evaluations, model_id)

    # Build prompt and call evaluator LLM (single batched call)
    user_prompt = build_evaluation_prompt(cited_claims, citation_lookup)

    client = InferenceClient(api_key=api_key)
    completion = client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": EVALUATION_SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    raw_response = completion.choices[0].message.content

    # Parse JSON response with fallbacks
    eval_data = _parse_evaluation_response(raw_response)

    # Map evaluation results back to their original claims
    for i, claim in enumerate(cited_claims):
        claim_idx = i + 1  # 1-indexed to match prompt numbering

        # Strategy 1: match by claim_index (1-based — most common)
        match = next((item for item in eval_data if item.get("claim_index") == claim_idx), None)

        # Strategy 2: match by claim_index (0-based — some models use this)
        if match is None:
            match = next((item for item in eval_data if item.get("claim_index") == i), None)

        # Strategy 3: positional fallback — use the i-th result in order
        if match is None and i < len(eval_data):
            match = eval_data[i]

        if match:
            support = match.get("support_level", "partial")
            if support not in ("supported", "partial", "unsupported", "contradicted"):
                support = "partial"
            conf = float(match.get("confidence", 0.5))
            conf = max(0.0, min(1.0, conf))
            evaluations.append(ClaimEvaluation(
                claim_text=claim["text"],
                citation_ids=claim["citation_ids"],
                support_level=support,
                confidence=conf,
                reasoning=match.get("reasoning", "No reasoning provided."),
            ))
        else:
            evaluations.append(ClaimEvaluation(
                claim_text=claim["text"],
                citation_ids=claim["citation_ids"],
                support_level="partial",
                confidence=0.5,
                reasoning="Evaluator did not return a result for this claim.",
            ))

    return _build_result(evaluations, model_id)

print("✅ Evaluation functions loaded")

### 5. Top-Level Entry Point

`run_evaluation()` is the single function the pipeline calls. It:
1. Calls `extract_claims()` to split the response
2. Calls `evaluate_claims()` to score each claim
3. Returns an `EvaluationResult`, or `None` if anything goes wrong

The `try/except` wrapper is important: evaluation is a bonus feature — if the evaluator API is down or the model fails, the pipeline should still return the answer.

In [ ]:
def run_evaluation(
    response_text: str,
    citation_lookup: Dict[str, Dict],
    model_id: str,
    api_key: str,
    temperature: float = 0.1,
    max_tokens: int = 1000,
) -> Optional[EvaluationResult]:
    """
    Top-level function: extract claims, evaluate, return results.

    This is the single entry point called from the pipeline.
    Never crashes — returns None on failure.

    Args:
        response_text: Full LLM response with citation markers
        citation_lookup: Sources dict from link_citations()["sources"]
        model_id: Model to use for evaluation
        api_key: HF API key
        temperature: Evaluation temperature (default 0.1 for determinism)
        max_tokens: Max tokens for evaluation response

    Returns:
        EvaluationResult or None if evaluation fails entirely
    """
    try:
        claims = extract_claims(response_text)

        if not claims:
            print("  ⚠️  No claims extracted from response. Skipping evaluation.")
            return None

        cited_count   = sum(1 for c in claims if c["citation_ids"])
        uncited_count = sum(1 for c in claims if not c["citation_ids"])
        print(f"  Extracted {len(claims)} claims ({cited_count} cited, {uncited_count} uncited)")

        result = evaluate_claims(
            claims=claims,
            citation_lookup=citation_lookup,
            model_id=model_id,
            api_key=api_key,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return result

    except Exception as e:
        print(f"  ⚠️  Evaluation failed: {e}")
        return None

print("✅ run_evaluation() loaded")

### 6. Updated Answer Generation with Evaluation

We extend `generate_answer_with_citations()` from Phase 2 by adding an optional `enable_evaluation` flag.

When `enable_evaluation=True`, the function runs `run_evaluation()` on the linked response before formatting. The evaluation result is then included in the returned dictionary and rendered in the formatted output.

```
[Phase 2 flow]
generate_answer()  →  link_citations()  →  citation_formatter()

[Phase 3 addition]
generate_answer()  →  link_citations()  →  run_evaluation()  →  citation_formatter()
                                                  ↑
                                         (only when enabled)
```

Setting `enable_evaluation=False` (the default) gives identical behaviour to Phase 2 — no extra latency, no regressions.

In [ ]:
def format_evaluation(evaluation: EvaluationResult) -> str:
    """
    Format an EvaluationResult as a plain-text confidence report.
    Appended to the citation_formatter output.
    """
    LEVEL_LABELS = {
        "supported":    "SUPPORTED",
        "partial":      "PARTIAL",
        "unsupported":  "UNSUPPORTED",
        "contradicted": "CONTRADICTED",
        "uncited":      "UNCITED",
    }

    text  = "=" * 60 + "\n"
    text += f"CONFIDENCE EVALUATION (Overall: {evaluation.overall_confidence:.2f})\n"
    text += "=" * 60 + "\n\n"

    for i, claim in enumerate(evaluation.claim_evaluations, 1):
        label = LEVEL_LABELS.get(claim.support_level, claim.support_level.upper())
        conf_str = f"{claim.confidence:.2f}" if claim.support_level != "uncited" else "N/A"
        text += f"[{i}] {label} ({conf_str})\n"
        text += f"    Claim: {claim.claim_text[:120]}\n"
        text += f"    Reason: {claim.reasoning}\n\n"

    return text


def generate_answer_with_citations_phase3(
    prompt: str,
    model_id: str,
    api_key: str,
    retrieved_chunks: List[Document],
    temperature: float = 0.3,
    max_tokens: int = 500,
    enable_evaluation: bool = True,
) -> dict:
    """
    Generate an answer with citations and optional LLM-as-a-judge evaluation.

    Args:
        prompt: Citation-aware prompt string
        model_id: HuggingFace model ID (used for both generation and evaluation)
        api_key: HuggingFace API key
        retrieved_chunks: Retrieved documents with citation metadata
        temperature: Sampling temperature for generation
        max_tokens: Maximum tokens in generation response
        enable_evaluation: Whether to run Phase 3 uncertainty evaluation

    Returns:
        Dictionary with response, citations, formatted output, and evaluation
    """
    # Step 1: Generate answer (same as Phase 2)
    response = generate_answer(
        prompt=prompt,
        model_id=model_id,
        api_key=api_key,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    # Step 2: Link citations (same as Phase 2)
    linker = CitationLinker()
    linked_response = linker.link_citations(response, retrieved_chunks)

    # Step 3: Run evaluation (NEW in Phase 3)
    evaluation = None
    if enable_evaluation:
        print("\n  [Phase 3] Running uncertainty evaluation...")
        evaluation = run_evaluation(
            response_text=linked_response["response"],
            citation_lookup=linked_response["sources"],
            model_id=model_id,
            api_key=api_key,
        )
        if evaluation:
            print(f"  [Phase 3] Overall confidence: {evaluation.overall_confidence:.2f}")

    # Step 4: Format output (citations + optional confidence report)
    formatted = citation_formatter(
        linked_response["response"],
        linked_response["citations"],
        linked_response["sources"],
    )
    if evaluation:
        formatted += "\n" + format_evaluation(evaluation)

    return {
        "raw_response":    response,
        "linked_response": linked_response,
        "formatted":       formatted,
        "evaluation":      evaluation.to_dict() if evaluation else None,
    }

print("✅ generate_answer_with_citations_phase3() loaded")

### 7. Complete Phase 3 RAG Pipeline

Now we put everything together. The Phase 3 pipeline is identical to Phase 2, with one new step inserted between citation linking and output:

```
User Question
    ↓
Step 1: retrieval_with_citations()          → Retrieve chunks + citation metadata
    ↓
Step 2: format_citation_aware_prompt()      → Build prompt with citation guidelines
    ↓
Step 3: generate_answer_with_citations_phase3()
         ├── generate_answer()              → LLM generates cited response
         ├── link_citations()               → Extract & link [c3.0.0] markers
         ├── run_evaluation()  ← NEW        → Score each claim against its source
         └── citation_formatter()           → Render answer + sources + confidence
    ↓
Final Output: Answer + Sources + Confidence Evaluation
```

In [ ]:
def rag_pipeline_phase3(vector_store, user_question, system_prompt):
    """
    Complete Phase 3 RAG pipeline: Retrieve → Format → Generate + Cite + Evaluate

    Args:
        vector_store: FAISS vector store with embeddings
        user_question: User's question
        system_prompt: System instructions for the LLM

    Returns:
        Dictionary with answer, citations, formatted output, and evaluation
    """
    print("\n" + "="*60)
    print("PHASE 3 RAG PIPELINE EXECUTION")
    print("="*60)

    try:
        # STEP 1: RETRIEVAL WITH CITATIONS
        print("\n[Step 1/3] Retrieving relevant chunks with citation metadata...")
        retrieved_chunks = retrieval_with_citations(
            vector_store,
            user_question,
            k=TOP_K_RESULTS
        )
        print(f"✅ Retrieved {len(retrieved_chunks)} chunks")

        # STEP 2: CITATION-AWARE PROMPT FORMATTING
        print("\n[Step 2/3] Formatting citation-aware prompt...")
        prompt = format_citation_aware_prompt(user_question, retrieved_chunks, system_prompt)
        print("✅ Prompt formatted")

        # STEP 3: GENERATE + EVALUATE
        print("\n[Step 3/3] Generating answer and running evaluation...")
        result = generate_answer_with_citations_phase3(
            prompt=prompt,
            model_id=HF_MODEL_ID,
            api_key=HF_TOKEN,
            retrieved_chunks=retrieved_chunks,
            temperature=LLM_TEMPERATURE,
            max_tokens=LLM_MAX_TOKENS,
            enable_evaluation=True,
        )
        print("✅ Answer generated and evaluated")

        return {
            "question":    user_question,
            "answer":      result["linked_response"]["response"],
            "citations":   result["linked_response"]["citations"],
            "sources":     retrieved_chunks,
            "formatted":   result["formatted"],
            "evaluation":  result["evaluation"],
            "source_count": len(retrieved_chunks),
        }

    except Exception as e:
        print(f"\n❌ Error in Phase 3 pipeline: {e}")
        raise

print("✅ rag_pipeline_phase3() loaded and ready")

### Complete Phase 3

All the pieces are in place. Add a question and run the cell below to get back:
- The answer with inline citations
- A source list (same as Phase 2)
- A per-claim confidence evaluation showing which claims are supported by the cited sources

In [ ]:
question = "What did Prime Minister Wong say about AI with regards to Singapore?"
result = rag_pipeline_phase3(vector_store, question, SYSTEM_PROMPT)

print("\n" + "="*60)
print("CHATBOT RESPONSE")
print("="*60)
print(f"\nQuestion: {result['question']}\n")
print(result["formatted"])

# Summary of evaluation scores
if result.get("evaluation"):
    eval_data = result["evaluation"]
    print("="*60)
    print("CONFIDENCE SUMMARY")
    print("="*60)
    print(f"Overall Confidence : {eval_data['overall_confidence']:.2f}")
    print(
        f"Claims             : {eval_data['total_claims']} total  |  "
        f"{eval_data['supported']} supported  |  "
        f"{eval_data['partial']} partial  |  "
        f"{eval_data['unsupported']} unsupported  |  "
        f"{eval_data['uncited']} uncited"
    )

---

### EXERCISE: Fact-Check PM Wong's AI Statements

Now it's your turn! Test the fact-checking system with different statements about what PM Wong said.

**Your task:**
Below are some *sample claims* about Singapore's AI strategy. For each one:
1. Query the chatbot to get its answer
2. Compare the LLM's confidence score with your own assessment
3. Check if the citation IDs are accurate (do they actually support the claim?)

**Example statements to test:**
- "Singapore will invest in AI to overcome structural constraints"
- "PM Wong said AI should be deployed responsibly and at speed"
- "The government will begin a large-scale deployment of AI to replace human workers"
- "AI is the government's top priority in 2026" (Is this explicitly stated or inferred?)
- "Every Singaporean will be given free access to premium AI models"
- "AI will continue to be prioritized by the government in future budgets."

**Challenge questions:**
1. Which claims have the highest confidence scores? Why?
2. Did you find any cases where the LLM cited sources correctly but the confidence was low?
3. Are there claims in the budget that the chatbot struggles to find or evaluate?

In [ ]:
# EXERCISE: Fact-check PM Wong's statements
test_query = # FIX ME: Replace this query with one of the suggestions above, or write your own!

print(f"Testing: {test_query}\n")
result = rag_pipeline_phase3(vector_store, test_query, SYSTEM_PROMPT)

print("\n" + "="*60)
print("CHATBOT RESPONSE WITH CONFIDENCE EVALUATION")
print("="*60)
print(f"\nQuestion: {result['question']}\n")
print(result["formatted"])

# Analysis: Print just the confidence scores for easy comparison
if result.get("evaluation"):
    eval_data = result["evaluation"]
    print("\n" + "="*60)
    print("QUICK EVALUATION SUMMARY")
    print("="*60)
    print(f"Overall Confidence: {eval_data['overall_confidence']:.2f}/1.00")
    print(f"\nClaim breakdown:")
    print(f"  ✅ Supported     : {eval_data['supported']} claims")
    print(f"  🟡 Partial       : {eval_data['partial']} claims")
    print(f"  ❌ Unsupported   : {eval_data['unsupported']} claims")
    print(f"  🔄 Contradicted  : {eval_data['contradicted']} claims")
    print(f"  ⚠️  Uncited       : {eval_data['uncited']} claims")

---

## 📚 Additional Resources & Deep Dives

Congratulations on building your first citation-aware PDF ChatBot! To continue your journey into the world of LLMs and RAG, check out these curated resources.

### 🔍 Understanding RAG (Retrieval-Augmented Generation)

* **[Building a Simple Talk-to-PDF Chatbot](https://anuragkmr.medium.com/building-a-simple-talk-to-pdf-chatbot-338f7ef1231f)** – A clear walkthrough of the core logic behind PDF-based conversational agents.
* **[IBM: What is Retrieval-Augmented Generation?](https://research.ibm.com/blog/retrieval-augmented-generation-rag)** – A high-level conceptual overview of why RAG is necessary for reducing hallucinations.
* **[Pinecone: The RAG Learning Center](https://www.pinecone.io/learn/retrieval-augmented-generation/)** – Comprehensive guides on vector databases, similarity search, and retrieval strategies.
* **[LangChain Documentation](https://python.langchain.com/docs/get_started/introduction)** – The industry-standard framework used in this workshop. Explore their guides on "Chains" and "Agents."

### 🛠️ Advanced Tools & Frameworks

* **[Hugging Face NLP Course](https://huggingface.co/learn/nlp-course/)** – Learn how the underlying Transformer models and embeddings (like the `all-MiniLM-L6-v2` we used) actually work.
* **[LlamaIndex](https://www.llamaindex.ai/)** – A specialized data framework for LLM applications that excels at complex document indexing and retrieval.
* **[FAISS (Facebook AI Similarity Search)](https://engineering.fb.com/2017/03/29/ml-applications/faiss-a-library-for-efficient-similarity-search/)** – Detailed look at the engine powering our vector store.

### 📝 Engineering & Best Practices

* **[Mastering RAG Citations](https://www.tensorlake.ai/blog/rag-citations)** – A technical deep dive into ensuring your LLM points to the exact source of its information.
* **[Quantifying LLM Uncertainty & Confidence Scores](https://medium.com/capgemini-invent-lab/quantifying-llms-uncertainty-with-confidence-scores-6bb8a6712aa0)** – Learn how to implement confidence scores so your bot can signal its level of certainty.
* **[Prompt Engineering Guide](https://www.promptingguide.ai/)** – Master the art of "System Prompts" to make your bot more accurate and professional.
* **[Chunking Strategies for RAG](https://www.pinecone.io/learn/chunking-strategies/)** – A deep dive into why `chunk_size` and `overlap` matter for retrieval quality.
* **[Another Guide to Chunking](https://community.databricks.com/t5/technical-blog/the-ultimate-guide-to-chunking-strategies-for-rag-applications/ba-p/113089)** - This blog provides more code-based and practical approach to understanding chunking approaches and the earlier mentioned chunking parameters.
* **[RAG Evaluation (RAGAS)](https://docs.ragas.io/en/stable/)** – Frameworks for measuring the performance and accuracy of your RAG pipeline.

### 💡 Community & Projects

* **[DeepLearning.AI Short Courses](https://www.deeplearning.ai/short-courses/)** – Free, short courses on building with LangChain, taught by industry leaders.
* **[GitHub: "Awesome RAG"](https://github.com/jxzhangjhu/Awesome-LLM-RAG)** – A curated list of RAG papers, projects, and frameworks. The perfect place to find your next deep-dive topic or open-source contribution.

---
